# GeoSR-4 — DINOv3 sat493m perceptual-backbone run (Kaggle GPU)

Kaggle version of `train_swinir_dino3_colab.ipynb` -- only the DINOv3 sat493m run (D042), for when Colab keeps disconnecting/hitting usage limits before the ~4.5 hour run (30 epochs, ~9 min/epoch) finishes.

`facebook/dinov3-vitl16-pretrain-sat493m` is ViT-L (300M params), satellite-pretrained (SAT-493M dataset) -- gated access, approved for this project already (D042). `--batch-size 4` is a conservative starting guess against T4's 15GB -- reduce further if it OOMs (see D024/D025).

**Before running, in the notebook's right-hand Settings panel:**
- Accelerator: **GPU T4 x2** or **P100** -- either is fine, `--amp` is already wired in for T4.
- Internet: ON (needed for git clone, dataset + model download). If you hit `Could not resolve host`, your Kaggle account likely isn't phone-verified yet (Settings → Account → Phone Verification).

Kaggle gives ~30 GPU-hours/week -- this single ~4.5 hour run fits comfortably, and (unlike the Colab free tier) Kaggle sessions don't get cut off by a shared-GPU usage limit mid-run, only by the fixed 12-hour session cap and the weekly quota.

In [ ]:
!nvidia-smi

## 1. Clone the repo and install dependencies
Kaggle's default image already has PyTorch with CUDA -- only the packages it's missing get installed.

In [ ]:
!git clone https://github.com/Vijay6923/GeoSR-4.git
%cd GeoSR-4
!pip install -q rasterio huggingface_hub scikit-image torchvision transformers

## 2. Download the dataset (cross-sensor split only, ~2.1 GB)

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, os

zip_path = hf_hub_download(
    repo_id="isp-uv-es/SEN2NAIP",
    repo_type="dataset",
    filename="cross-sensor/cross-sensor.zip",
    local_dir="ml/datasets/raw/sen2naip",
)

extract_dir = "ml/datasets/raw/sen2naip/cross-sensor/extracted"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_dir)

print("extracted to", extract_dir)

## 3. Hugging Face login (required -- gated model)
**Recommended on Kaggle**: use a Kaggle Secret instead of pasting a token here -- left sidebar → "Add-ons" → "Secrets" → add one named `HF_TOKEN` with your token as the value, then attach it to this notebook. The cell below reads it from there automatically. (If you skip the secret, `login()` still works with its own interactive prompt.)

Must be the HF account that was granted access to `facebook/dinov3-vitl16-pretrain-sat493m` -- generate the token from *that* account's [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (Read scope is enough).

In [ ]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=token)
    print("logged in via Kaggle secret")
except Exception as e:
    print("no Kaggle secret found (or failed to read one) -- falling back to interactive login:", e)
    login()

## 4. Train: DINOv3 sat493m perceptual backbone
Checkpoints save every epoch to `experiments/swinir_dino3_sat_quality/` -- if this session gets interrupted, **Save Version** first (top right) so `/kaggle/working/` contents (including checkpoints saved so far) are preserved as that version's output. To resume from one: add it as a Kaggle Dataset input to a new notebook, then pass its path to `--resume-from` (see section 4b).

In [ ]:
!python ml/training/train_swinir.py \
  --epochs 30 \
  --batch-size 4 \
  --embed-dim 60 \
  --depths 2,2,2,2 \
  --num-heads 6 \
  --window-size 11 \
  --lr 1e-4 \
  --lambda-perceptual 0.01 \
  --perceptual-backbone dino \
  --dino-model-id facebook/dinov3-vitl16-pretrain-sat493m \
  --amp \
  --checkpoint-dir experiments/swinir_dino3_sat_quality \
  --log-every 20

!python ml/evaluation/evaluate_checkpoint.py --checkpoint experiments/swinir_dino3_sat_quality/swinir_epoch29.pt \
  --model-type swinir --embed-dim 60 --depths 2,2,2,2 --num-heads 6 --window-size 11

## 4b. Resuming after an interruption (skip if section 4 just ran fine)
Only run this if a previous attempt got interrupted -- point `--resume-from` at the last checkpoint you have
(e.g. from a Kaggle Dataset you added as input, under `/kaggle/input/<dataset-name>/`, or still present in
`/kaggle/working/` if this is the *same* session). Only the model weights resume, not the optimizer's momentum
state -- a documented simplification for finishing an interrupted run, see `--resume-from`'s help text in
`train_swinir.py`.

In [ ]:
RESUME_FROM = "experiments/swinir_dino3_sat_quality/swinir_epoch23.pt"  # <-- edit to your actual last checkpoint path

!python ml/training/train_swinir.py \
  --epochs 30 \
  --batch-size 4 \
  --embed-dim 60 \
  --depths 2,2,2,2 \
  --num-heads 6 \
  --window-size 11 \
  --lr 1e-4 \
  --lambda-perceptual 0.01 \
  --perceptual-backbone dino \
  --dino-model-id facebook/dinov3-vitl16-pretrain-sat493m \
  --amp \
  --checkpoint-dir experiments/swinir_dino3_sat_quality \
  --resume-from {RESUME_FROM} \
  --log-every 20

!python ml/evaluation/evaluate_checkpoint.py --checkpoint experiments/swinir_dino3_sat_quality/swinir_epoch29.pt \
  --model-type swinir --embed-dim 60 --depths 2,2,2,2 --num-heads 6 --window-size 11

## 5. Get the checkpoint out
Copies it to `/kaggle/working/`, downloadable from the notebook's **Output** tab once you save a version.

In [ ]:
import shutil
shutil.copy("experiments/swinir_dino3_sat_quality/swinir_epoch29.pt", "/kaggle/working/swinir_dino3_sat_epoch29.pt")
print("copied -- visible in the Output tab once you save a version of this notebook")